In [1]:
import os
os.chdir(r'C:\Users\Klara\retail-intelligence')

import pandas as pd

eval_table = pd.read_csv('data/processed/final_evaluation_table.csv')
print(eval_table.shape)
print(eval_table.columns.tolist())

(20, 22)
['StockCode', 'Prophet_MAE', 'Prophet_RMSE', 'Prophet_MAPE', 'Naive_MAE', 'Naive_RMSE', 'Naive_MAPE', 'SeasonalNaive_MAE', 'SeasonalNaive_RMSE', 'SeasonalNaive_MAPE', 'BeatNaive', 'BeatSeasonalNaive', 'DataQualityFlag', 'MeanRelativeUncertainty', 'MeanForecast', 'TrendDirection', 'TrendChangeUnits', 'TrendChangeVsMean%', 'RiskStatus', 'Description', 'ImprovementOverNaive%', 'ImprovementOverSeasonal%']


In [2]:
reliable = eval_table[eval_table['DataQualityFlag'] == 'OK'].sort_values('Prophet_MAE')

print(reliable[['StockCode', 'Description', 'Prophet_MAE', 'Naive_MAE', 
                 'SeasonalNaive_MAE', 'BeatNaive', 'BeatSeasonalNaive',
                 'ImprovementOverNaive%', 'ImprovementOverSeasonal%',
                 'MeanRelativeUncertainty', 'TrendDirection', 'RiskStatus']].to_string(index=False))

StockCode                        Description  Prophet_MAE  Naive_MAE  SeasonalNaive_MAE  BeatNaive  BeatSeasonalNaive  ImprovementOverNaive%  ImprovementOverSeasonal%  MeanRelativeUncertainty TrendDirection RiskStatus
    22386            JUMBO BAG PINK POLKADOT       143.87     107.62             256.00      False               True                  -33.7                      43.8                    270.8        Growing  Low Stock
    21212    PACK OF 72 RETROSPOT CAKE CASES       161.28     213.12             402.12       True               True                   24.3                      59.9                    217.2      Declining   Adequate
    84946       ANTIQUE SILVER T-LIGHT GLASS       171.00     171.00             195.62      False               True                    0.0                      12.6                    210.2        Growing  Low Stock
   85099F               JUMBO BAG STRAWBERRY       176.87     140.25             171.25      False              False           

In [3]:
print("Win rate:")
print(f"  Beats naive: {reliable['BeatNaive'].sum()}/{len(reliable)}")
print(f"  Beats seasonal naive: {reliable['BeatSeasonalNaive'].sum()}/{len(reliable)}")

print("\nAverage MAE:")
print(f"  Prophet: {reliable['Prophet_MAE'].mean():.1f}")
print(f"  Naive: {reliable['Naive_MAE'].mean():.1f}")
print(f"  Seasonal Naive: {reliable['SeasonalNaive_MAE'].mean():.1f}")

print("\nImprovement over naive - mean: {:.1f}% | median: {:.1f}%".format(
    reliable['ImprovementOverNaive%'].mean(), reliable['ImprovementOverNaive%'].median()
))
print("Improvement over seasonal naive - mean: {:.1f}% | median: {:.1f}%".format(
    reliable['ImprovementOverSeasonal%'].mean(), reliable['ImprovementOverSeasonal%'].median()
))

Win rate:
  Beats naive: 7/18
  Beats seasonal naive: 13/18

Average MAE:
  Prophet: 336.1
  Naive: 379.5
  Seasonal Naive: 546.4

Improvement over naive - mean: -13.1% | median: -3.2%
Improvement over seasonal naive - mean: -1.6% | median: 27.6%


## Day 5 — Notes & Observations

### What each cell did

**Data loading cell** — loaded `final_evaluation_table.csv`, the merged output of all four Day 2-4 CSVs (baseline comparison, uncertainty metrics, seasonality analysis, inventory risk) into one table, 20 products x 22 columns.

**Full sorted table** — printed all 18 OK products sorted by Prophet_MAE, with MAE, improvement %, uncertainty, trend, and inventory status together in one view.

**Summary statistics cell** — computed win rate, average MAE, and mean/median improvement directly in the notebook.

### Fix applied earlier today: MAPE near-zero blowup

`calculate_mape` originally excluded only weeks where actual demand was exactly 0. Weeks with very small non-zero actuals (e.g. 1-2 units) still caused extreme percentage errors when divided into, inflating some products' MAPE into the thousands (21915 originally showed 3517%). Fixed by excluding weeks below a 5-unit threshold instead of just exact zero. Prophet's average MAPE (OK products) dropped from a misleading 292.2% to a real 122.9%; 21915 individually dropped from 3517% to 470.2%.

### What the numbers actually say (18 OK products, n=18)

- **Win rate:** Prophet beats naive on 7/18 products (39%); beats seasonal naive   on 13/18 products (72%)
- **Average MAE:** Prophet 336.1 vs Naive 379.5 vs Seasonal Naive 546.4 - Prophet has the lowest average absolute error of the three
- **Improvement over naive:** mean -13.1%, median -3.2% - roughly a wash on the typical product, slightly favoring naive
- **Improvement over seasonal naive:** mean -1.6%, median +27.6% - a large gap between mean and median reveals the mean is dragged down by a few products with extreme percentage swings; on a typical product, Prophet meaningfully outperforms seasonal naive

**Bottom line:** "Prophet's trend-only model modestly underperforms a naive repeat-last-week baseline on the typical product, but clearly outperforms a seasonal naive baseline (same week last year) on 72% of products, with the median product showing a 27.6% error reduction - the gap between mean (-1.6%) and median (+27.6%) improvement over seasonal naive is itself a finding: a few products with large percentage swings pull the mean down even though most products see a real, meaningful improvement."

### Products flagged for Week 5 walk-forward validation (not fixed today)

**21977** (Day 4 finding): Prophet's trend line overshot a flat/declining test period, losing badly to naive. Cause: "Growing" trend with no seasonality to temper it. Not fixed today since a code change would risk tuning to this single 8-week window; deferred to Week 5.

**15036** (today's finding): ImprovementOverSeasonal% of -312.3%, the largest negative outlier in the table. Seasonal naive's MAE here (43.5) is unusually low - either a genuinely strong seasonal pattern or a lucky single-window match. Not a coding bug (the math is correct), just a noisy single-window result - same reasoning as 21977, deferred to Week 5's walk-forward validation across multiple windows.